In [1]:
from resources.vectorstore import get_vectorstore

vectorstore = get_vectorstore()

def retrieve_node_no_threshold(state):
    print("--- NOTEBOOK RETRIEVE (NO THRESHOLD) ---")
    question = state["question"]

    docs = vectorstore.similarity_search(question, k=10)

    return {
        "documents": docs,
        "steps": ["retrieve_no_threshold"]
    }


In [2]:
def relevance_grader_node_sop(state):
    print("--- NOTEBOOK RELEVANCE (SOP MODE) ---")

    docs = state["documents"]

    # If we retrieved anything at all, treat as relevant for demo
    if docs:
        return {
            "documents": docs,
            "is_relevant": "yes",
            "steps": ["sop_relevance_override"]
        }

    return {
        "documents": [],
        "is_relevant": "no",
        "steps": ["sop_relevance_override"]
    }


In [3]:
from langgraph.graph import StateGraph, START, END
from graph.state import GraphState

from graph.nodes.relevance import relevance_grader_node
from graph.nodes.generate import generate_node
from graph.nodes.hallucination import hallucination_grader_node
from graph.nodes.retry_generate import retry_generate_node
from graph.nodes.improve_kb import improve_kb
from graph.routers import decide_after_relevance, decide_final_step


In [4]:
workflow_nb = StateGraph(GraphState)

workflow_nb.add_node("retrieve", retrieve_node_no_threshold)
workflow_nb.add_node("grade_relevance", relevance_grader_node_sop)
workflow_nb.add_node("generate", generate_node)
workflow_nb.add_node("grade_hallucination", hallucination_grader_node)
workflow_nb.add_node("retry_generate", retry_generate_node)
workflow_nb.add_node("improve_kb", improve_kb)

workflow_nb.add_edge(START, "retrieve")
workflow_nb.add_edge("retrieve", "grade_relevance")

workflow_nb.add_conditional_edges(
    "grade_relevance",
    decide_after_relevance,
    {
        "generate": "generate",
        "improve_kb": "improve_kb",
    },
)


workflow_nb.add_edge("improve_kb", "retrieve")
workflow_nb.add_edge("generate", "grade_hallucination")

workflow_nb.add_conditional_edges(
    "grade_hallucination",
    decide_final_step,
    {
        "retry_generate": "retry_generate",
        "improve_kb": "improve_kb",
        "finalize": END,
    },
)

workflow_nb.add_edge("retry_generate", "grade_hallucination")

app_nb = workflow_nb.compile()


In [ ]:
state = {
    "question": "",
    "documents": [],
    "retry_generation_count": 0,
    "kb_retry_count": 0,
    "kb_enriched": True,
    "steps": []
}

result = app_nb.invoke(state)

print("ANSWER:")
print(result["generation"])

print("\nSTEPS:")
print(result["steps"])


# FOR SOP LOADING


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from resources.vectorstore import get_vectorstore
from tools.write_to_vector_db import write_to_vector_db


In [2]:
def load_and_store_sop(
    pdf_path: str,
    sop_name: str,
    version: str,
    owner: str = "BFS Ops",
    chunk_size: int = 1200,
    chunk_overlap: int = 200,
):
    """
    Load a multi-page SOP PDF, chunk it safely, and store it in ChromaDB.

    Returns:
        stored_chunks (int): Number of new chunks written to the vector DB
    """

    print("📄 Loading SOP PDF...")
    loader = PyPDFLoader(pdf_path)
    raw_docs = loader.load()

    print(f"   → Loaded {len(raw_docs)} pages")

    # Add SOP metadata
    for doc in raw_docs:
        doc.metadata.update({
            "source": "sop",
            "doc_type": "procedure",
            "sop_name": sop_name,
            "version": version,
            "owner": owner,
            "page": doc.metadata.get("page"),
            "file": pdf_path,
        })

    print("✂️ Chunking SOP content...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " "]
    )

    chunked_docs = []
    for doc in raw_docs:
        chunks = splitter.split_text(doc.page_content)
        for chunk in chunks:
            chunked_docs.append(
                Document(
                    page_content=chunk,
                    metadata=doc.metadata
                )
            )

    print(f"   → Created {len(chunked_docs)} chunks")

    # Get vectorstore (singleton)
    vectorstore = get_vectorstore()

    print("📥 Writing SOP chunks to vector DB...")
    stored_chunks = write_to_vector_db(
        docs=chunked_docs,
        vectorstore=vectorstore
    )

    print("✅ SOP ingestion complete")
    print(f"   → Stored new chunks: {stored_chunks}")

    return stored_chunks


In [3]:
stored = load_and_store_sop(
    pdf_path=r"C:\Users\ddev\Documents\projects\BFS Data Agent\Updated Names SOP.pdf",
    sop_name="Banking & Financial Services Data Processing SOP",
    version="v1.2",
    owner="Data Platform Team"
)


📄 Loading SOP PDF...
   → Loaded 16 pages
✂️ Chunking SOP content...
   → Created 20 chunks
📥 Writing SOP chunks to vector DB...
✅ SOP ingestion complete
   → Stored new chunks: 36
